# India Crypto Trading Tax Calculator (2026)


In [ ]:
# Install dependencies
!pip install -q requests pandas matplotlib numpy


# India Crypto Trading Tax Calculator (2026)

This notebook helps Indian crypto traders calculate capital gains tax and TDS (Tax Deducted at Source) under India's 30% flat tax regime on virtual digital assets. Based on current regulatory framework for Indian traders.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

# Define sample crypto trades
# Format: Symbol, Entry Price (INR), Exit Price (INR), Quantity, Entry Date, Exit Date
trades = [
    {'symbol': 'BTC', 'entry_price': 2500000, 'exit_price': 3200000, 'quantity': 0.5, 'entry_date': '2025-01-01', 'exit_date': '2026-01-15'},
    {'symbol': 'ETH', 'entry_price': 150000, 'exit_price': 210000, 'quantity': 2, 'entry_date': '2025-03-01', 'exit_date': '2026-02-20'},
    {'symbol': 'USDT', 'entry_price': 83, 'exit_price': 85, 'quantity': 100000, 'entry_date': '2025-06-01', 'exit_date': '2026-01-10'},
]

print("Sample Crypto Trades for Analysis:")
for i, trade in enumerate(trades, 1):
    print(f"{i}. {trade['symbol']}: {trade['quantity']} units @ {trade['entry_price']} INR -> {trade['exit_price']} INR")

In [ ]:
# Calculate capital gains and tax liability
TAX_RATE = 0.30  # 30% flat tax on crypto gains

tax_results = []

for trade in trades:
    entry_price = trade['entry_price']
    exit_price = trade['exit_price']
    quantity = trade['quantity']
    symbol = trade['symbol']
    
    # Calculate total investment and proceeds
    total_investment = entry_price * quantity
    total_proceeds = exit_price * quantity
    
    # Calculate capital gain/loss
    capital_gain = total_proceeds - total_investment
    
    # Apply 30% tax (only on gains, not losses)
    if capital_gain > 0:
        tax_liability = capital_gain * TAX_RATE
    else:
        tax_liability = 0  # Losses don't incur tax
    
    # Net profit after tax
    net_profit = capital_gain - tax_liability
    
    # Parse dates to calculate holding period
    entry_date = datetime.strptime(trade['entry_date'], '%Y-%m-%d')
    exit_date = datetime.strptime(trade['exit_date'], '%Y-%m-%d')
    holding_days = (exit_date - entry_date).days
    
    tax_results.append({
        'Symbol': symbol,
        'Entry Price (INR)': f"{entry_price:,.0f}",
        'Exit Price (INR)': f"{exit_price:,.0f}",
        'Quantity': quantity,
        'Investment (INR)': f"{total_investment:,.2f}",
        'Proceeds (INR)': f"{total_proceeds:,.2f}",
        'Capital Gain (INR)': f"{capital_gain:,.2f}",
        'Tax at 30% (INR)': f"{tax_liability:,.2f}",
        'Net Profit (INR)': f"{net_profit:,.2f}",
        'Holding Days': holding_days
    })

df_results = pd.DataFrame(tax_results)
print("\n=== TAX CALCULATION RESULTS ===")
print(df_results.to_string(index=False))

In [ ]:
# Calculate total portfolio tax metrics
print("\n=== PORTFOLIO SUMMARY ===")

total_investment = sum(trade['entry_price'] * trade['quantity'] for trade in trades)
total_proceeds = sum(trade['exit_price'] * trade['quantity'] for trade in trades)
total_gain = total_proceeds - total_investment
total_tax = max(0, total_gain * TAX_RATE)
net_portfolio_gain = total_gain - total_tax

print(f"Total Investment: ₹{total_investment:,.2f}")
print(f"Total Proceeds: ₹{total_proceeds:,.2f}")
print(f"Total Capital Gain: ₹{total_gain:,.2f}")
print(f"Total Tax Liability (30%): ₹{total_tax:,.2f}")
print(f"Net Profit After Tax: ₹{net_portfolio_gain:,.2f}")
print(f"Effective Tax Rate: {(total_tax / total_proceeds * 100) if total_proceeds > 0 else 0:.2f}%")

In [ ]:
# Visualization: Tax impact across trades
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Extract numeric values for plotting
symbols = [t['symbol'] for t in trades]
gains = [max(0, (t['exit_price'] - t['entry_price']) * t['quantity']) for t in trades]
taxes = [g * TAX_RATE for g in gains]
net_gains = [g - tax for g, tax in zip(gains, taxes)]

# Plot 1: Capital Gains vs Tax
ax = axes[0, 0]
x_pos = np.arange(len(symbols))
width = 0.35
ax.bar(x_pos - width/2, gains, width, label='Capital Gain', color='green', alpha=0.7)
ax.bar(x_pos + width/2, taxes, width, label='Tax (30%)', color='red', alpha=0.7)
ax.set_xlabel('Cryptocurrency')
ax.set_ylabel('Amount (INR)')
ax.set_title('Capital Gain vs Tax Liability by Trade')
ax.set_xticks(x_pos)
ax.set_xticklabels(symbols)
ax.legend()
ax.grid(axis='y', alpha=0.3)

# Plot 2: Net Gain After Tax
ax = axes[0, 1]
colors = ['green' if ng > 0 else 'red' for ng in net_gains]
ax.bar(symbols, net_gains, color=colors, alpha=0.7)
ax.set_xlabel('Cryptocurrency')
ax.set_ylabel('Net Profit After Tax (INR)')
ax.set_title('Net Profit After 30% Tax')
ax.grid(axis='y', alpha=0.3)

# Plot 3: Tax Breakdown (Pie)
ax = axes[1, 0]
if total_gain > 0:
    ax.pie([total_tax, net_portfolio_gain], labels=[f'Tax (₹{total_tax:,.0f})', f'Net Profit (₹{net_portfolio_gain:,.0f})'],
           autopct='%1.1f%%', colors=['#ff6b6b', '#51cf66'], startangle=90)
    ax.set_title('Portfolio: Tax vs Net Profit')
else:
    ax.text(0.5, 0.5, 'No Gains (No Tax)', ha='center', va='center')
    ax.set_title('Portfolio Tax Breakdown')

# Plot 4: ROI Comparison (Before and After Tax)
ax = axes[1, 1]
roi_before_tax = [(g / (t['entry_price'] * t['quantity'])) * 100 for t, g in zip(trades, gains)]
roi_after_tax = [(ng / (t['entry_price'] * t['quantity'])) * 100 for t, ng in zip(trades, net_gains)]
x_pos = np.arange(len(symbols))
width = 0.35
ax.bar(x_pos - width/2, roi_before_tax, width, label='Before Tax', color='blue', alpha=0.7)
ax.bar(x_pos + width/2, roi_after_tax, width, label='After Tax', color='orange', alpha=0.7)
ax.set_xlabel('Cryptocurrency')
ax.set_ylabel('ROI (%)')
ax.set_title('Return on Investment: Before vs After Tax')
ax.set_xticks(x_pos)
ax.set_xticklabels(symbols)
ax.legend()
ax.grid(axis='y', alpha=0.3)
ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)

plt.tight_layout()
plt.show()

print("\nGraphs generated successfully.")

## Key Notes on India Crypto Tax (2026)

- **30% Flat Tax**: All crypto capital gains are taxed at a fixed 30% rate
- **No Loss Offset**: Crypto losses cannot offset other income or gains
- **TDS on Exchanges**: Tax may be deducted at source by crypto exchanges when you trade
- **Holding Period**: No distinction between short-term and long-term gains
- **Reporting Requirement**: Report all crypto transactions in ITR (Income Tax Return)
- **Compliance**: Maintain records of all trades, exchanges used, and wallets

Always consult a qualified tax professional for your specific situation.

---

## Reference

[complete walkthrough](https://protraderdaily.com/crypto/can-i-do-crypto-trading-in-india)
